# Pipeline d'extraction GDELT - Bénin 2025

Ce notebook extrait depuis la base **GDELT v2** (Global Database of Events, Language and Tone) l'ensemble des données concernant le **Bénin** sur l'année **2025**, à travers trois pipelines distincts.

## Architecture globale

| Pipeline | Filtre appliqué | Fichier produit |
|---|---|---|
| **1 - Événements** | `ActionGeo_CountryCode == "BN"` | `gdelt_benin_2025_raw.csv` / `_clean.csv` |
| **2 - Mentions** | `GlobalEventID ∈ ids_bénin` | `gdelt_benin_2025_mentions.csv` |
| **3 - GKG** | `Locations` contient `"Benin"` | `gdelt_benin_2025_gkg.csv` |

---

# Pipeline 1 -  Événements GDELT (Export)

### Imports

In [ ]:
import pandas as pd
import requests
import zipfile
import io
from tqdm import tqdm

In [ ]:
url_master = "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt"

master = pd.read_csv(
    url_master,
    sep=" ",
    header=None,
    names=["taille", "md5", "url"],
    dtype=str      
)

print(f"Nombre total de fichiers dans le masterfilelist : {len(master)}")
print("\nAperçu des premières lignes :")
master.head(10)


master_2025 = master[
    master["url"].str.contains("2025") &
    master["url"].str.contains("export")
]

print(f"Nombre de fichiers export pour 2025 : {len(master_2025)}")
master_2025.head(5)

In [ ]:
url_test = master_2025["url"].iloc[0]
print("URL du fichier test :", url_test)

response = requests.get(url_test)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))

df_test = pd.read_csv(
    zip_file.open(zip_file.namelist()[0]),
    sep="\t",
    header=None,
    dtype=str
)

print(f"\nNombre de lignes : {len(df_test)}")
print(f"Nombre de colonnes : {len(df_test.columns)}")
df_test.head(3)

In [ ]:
colonnes = [
    "GlobalEventID", "SQLDATE", "MonthYear", "Year", "FractionDate",
    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
    "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
    "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode",
    "QuadClass", "GoldsteinScale", "NumMentions", "NumSources",
    "NumArticles", "AvgTone",
    "Actor1Geo_Type", "Actor1Geo_FullName", "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code", "Actor1Geo_ADM2Code", "Actor1Geo_Lat",
    "Actor1Geo_Long", "Actor1Geo_FeatureID",
    "Actor2Geo_Type", "Actor2Geo_FullName", "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code", "Actor2Geo_ADM2Code", "Actor2Geo_Lat",
    "Actor2Geo_Long", "Actor2Geo_FeatureID",
    "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code", "ActionGeo_ADM2Code", "ActionGeo_Lat",
    "ActionGeo_Long", "ActionGeo_FeatureID",
    "DATEADDED", "SOURCEURL"
]

df_test.columns = colonnes

df_test[["GlobalEventID", "SQLDATE", "Actor1Name", "Actor2Name",
         "ActionGeo_CountryCode", "GoldsteinScale", "AvgTone", "SOURCEURL"]].head(5)

df_benin_test = df_test[df_test["ActionGeo_CountryCode"] == "BN"]

print(f"Lignes totales dans le fichier : {len(df_test)}")
print(f"Lignes concernant le Bénin : {len(df_benin_test)}")

df_benin_test[["GlobalEventID", "SQLDATE", "Actor1Name", "Actor2Name",
               "GoldsteinScale", "AvgTone", "SOURCEURL"]]

In [ ]:
import time

mois_2025 = [
    "202501", "202502", "202503", "202504",
    "202505", "202506", "202507", "202508",
    "202509", "202510", "202511", "202512"
]

resultats_benin = []

for mois in mois_2025:

    fichiers_mois = master_2025[master_2025["url"].str.contains(mois)]


    compteur_benin = 0

    for url in tqdm(fichiers_mois["url"], desc=mois):
        try:
            response = requests.get(url, timeout=30)
            zip_file = zipfile.ZipFile(io.BytesIO(response.content))

            df = pd.read_csv(
                zip_file.open(zip_file.namelist()[0]),
                sep="\t",
                header=None,
                names=colonnes,
                dtype=str
            )

            df_bn = df[df["ActionGeo_CountryCode"] == "BN"]

            if len(df_bn) > 0:
                resultats_benin.append(df_bn)
                compteur_benin += len(df_bn)

        except Exception as e:

            print(f" Erreur sur {url} : {e}")
            continue

        time.sleep(0.1)

    print(f" {mois} terminé — {compteur_benin} événements Bénin trouvés")

print("\n Extraction terminée !")
print(f"Total événements Bénin : {sum(len(r) for r in resultats_benin)}")

In [ ]:

df_benin = pd.concat(resultats_benin, ignore_index=True)

print(f"Dimensions du dataset : {df_benin.shape}")
print(f"Lignes : {len(df_benin)}")
print(f"Colonnes : {len(df_benin.columns)}")

df_benin.head(5)
import os


dossier = "/content/drive/MyDrive/hackathon_isheero"
os.makedirs(dossier, exist_ok=True)
print(f"Dossier créé : {dossier}")


chemin_raw = f"{dossier}/gdelt_benin_2025_raw.csv"
df_benin.to_csv(chemin_raw, index=False)
print(f" Fichier brut sauvegardé : {chemin_raw}")

In [ ]:
print("1. Types de données par colonne :")
print(df_benin.dtypes.value_counts())

print("\n2. Valeurs manquantes par colonne :")
manquantes = df_benin.isnull().sum()
manquantes_pct = (manquantes / len(df_benin) * 100).round(1)
diagnostic = pd.DataFrame({
    "manquantes": manquantes,
    "pourcentage": manquantes_pct
}).sort_values("pourcentage", ascending=False)
print(diagnostic.head(15))

print("\n3. Doublons :")
print(f"Nombre de lignes dupliquées : {df_benin.duplicated().sum()}")

print("\n4. Aperçu des colonnes clés :")
print(df_benin[["SQLDATE", "GoldsteinScale", "AvgTone",
                 "NumArticles", "ActionGeo_CountryCode"]].describe())

In [ ]:
df_clean = df_benin.copy()

seuil = 0.90
cols_a_supprimer = [col for col in df_clean.columns
                    if df_clean[col].isnull().sum() / len(df_clean) > seuil]
df_clean = df_clean.drop(columns=cols_a_supprimer)
print(f"Colonnes supprimées ({len(cols_a_supprimer)}) : {cols_a_supprimer}")
print(f"Colonnes restantes : {len(df_clean.columns)}")

cols_numeriques = ["GoldsteinScale", "AvgTone", "NumMentions",
                   "NumSources", "NumArticles", "Actor1Geo_Lat",
                   "Actor1Geo_Long", "ActionGeo_Lat", "ActionGeo_Long"]

for col in cols_numeriques:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print(f"\nColonnes numériques converties ")

df_clean["SQLDATE"] = pd.to_datetime(df_clean["SQLDATE"], format="%Y%m%d")
print(f"SQLDATE converti en datetime ")

df_clean["mois"] = df_clean["SQLDATE"].dt.month
df_clean["mois_nom"] = df_clean["SQLDATE"].dt.strftime("%B")
df_clean["semaine"] = df_clean["SQLDATE"].dt.isocalendar().week

print(f"\nColonnes mois et semaine ajoutées ")
print(f"\nDimensions finales : {df_clean.shape}")

In [ ]:
print("Années présentes dans le dataset :")
print(df_clean["SQLDATE"].dt.year.value_counts())

df_clean = df_clean[df_clean["SQLDATE"].dt.year == 2025]

print(f"\nAprès filtre 2025 : {len(df_clean)} lignes")

---

# Pipeline 2 - Mentions GDELT

Les fichiers **mentions** tracent chaque article ayant cité un événement GDELT. Contrairement aux événements (filtrés géographiquement), les mentions sont filtrées par **identifiant d'événement** : on récupère toutes les sources médiatiques ayant couvert un événement déjà identifié comme concernant le Bénin.

**Clé de jointure avec Pipeline 1 :** `GlobalEventID`

### Colonnes des fichiers Mentions (16 colonnes)

| Colonne | Description |
|---|---|
| `GlobalEventID` | Clé de jointure avec le dataset événements |
| `EventTimeDate` | Horodatage de l'événement (format `YYYYMMDDHHMMSS`) |
| `MentionTimeDate` | Horodatage de publication de l'article |
| `MentionType` | Type de source (1 = web, 2 = citation, 3 = core, etc.) |
| `MentionSourceName` | Nom du média source |
| `MentionIdentifier` | URL de l'article |
| `SentenceID` | Numéro de la phrase dans laquelle l'événement est mentionné |
| `Confidence` | Score de confiance de l'extraction GDELT (0–100) |
| `MentionDocLen` | Longueur du document source en caractères |
| `MentionDocTone` | Tonalité globale du document |

> Les colonnes `Extra1` et `Extra2` (systématiquement vides dans GDELT v2) sont supprimées après lecture.  
> Les colonnes numériques (`MentionDocTone`, `Confidence`, `MentionDocLen`) sont castées avec `pd.to_numeric` après lecture.

In [ ]:
url_master = "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt"
master = pd.read_csv(
    url_master,
    sep=" ",
    header=None,
    names=["taille", "md5", "url"],
    dtype=str
)

mois_2025 = [
    "202501", "202502", "202503", "202504",
    "202505", "202506", "202507", "202508",
    "202509", "202510", "202511", "202512"
]

master_2025_mentions = master[
    master["url"].str.contains("2025") &
    master["url"].str.contains("mentions")
]

colonnes_mentions_correct = [
    "GlobalEventID",
    "EventTimeDate",
    "MentionTimeDate",
    "MentionType",
    "MentionSourceName",
    "MentionIdentifier",
    "SentenceID",
    "Actor1CharOffset",
    "Actor2CharOffset",
    "ActionCharOffset",
    "InRawText",
    "Confidence",
    "MentionDocLen",
    "MentionDocTone",
    "Extra1",
    "Extra2"
]

ids_benin = set(df_clean["GlobalEventID"].astype(str))
resultats_mentions = []

for mois in mois_2025:
    fichiers_mois = master_2025_mentions[
        master_2025_mentions["url"].str.contains(mois)
    ]
    print(f"\n {mois} — {len(fichiers_mois)} fichiers...")
    compteur = 0

    for url in tqdm(fichiers_mois["url"], desc=mois):
        try:
            response = requests.get(url, timeout=30)
            zip_file = zipfile.ZipFile(io.BytesIO(response.content))

            df = pd.read_csv(
                zip_file.open(zip_file.namelist()[0]),
                sep="\t",
                header=None,
                names=colonnes_mentions_correct,
                dtype=str
            )

            df_mn = df[df["GlobalEventID"].isin(ids_benin)]

            if len(df_mn) > 0:
                resultats_mentions.append(df_mn)
                compteur += len(df_mn)

        except Exception as e:
            print(f" Erreur : {e}")
            continue

        time.sleep(0.1)

    print(f" {mois} — {compteur} mentions Bénin trouvées")


df_mentions = pd.concat(resultats_mentions, ignore_index=True)
print(f"\n Total mentions brutes : {len(df_mentions)}")

# Nettoyage
df_mentions = df_mentions.drop(columns=["Extra1", "Extra2"])
df_mentions["MentionDocTone"] = pd.to_numeric(
    df_mentions["MentionDocTone"], errors="coerce"
)
df_mentions["Confidence"] = pd.to_numeric(
    df_mentions["Confidence"], errors="coerce"
)
df_mentions["MentionDocLen"] = pd.to_numeric(
    df_mentions["MentionDocLen"], errors="coerce"
)
df_mentions = df_mentions.drop_duplicates()
print(f" Après nettoyage : {len(df_mentions)} mentions")


chemin_mentions = f"{dossier}/gdelt_benin_2025_mentions.csv"
df_mentions.to_csv(chemin_mentions, index=False)
print(f" Mentions sauvegardées : {chemin_mentions}")

---

# Pipeline 3 - GKG (Global Knowledge Graph)

Le **GKG** est la couche sémantique de GDELT : pour chaque article indexé, il extrait les thèmes, personnes, organisations, lieux, citations et tonalités. C'est la source la plus riche pour l'analyse de contenu.

> **Différence de filtre :** Le GKG n'a pas de code pays standardisé. Le filtre se fait sur la présence du mot `"Benin"` dans les colonnes texte `Locations` et `V2Locations`. Cela peut introduire du bruit (ex : "Benin City" au Nigeria).

### Colonnes conservées du GKG (27 colonnes :  12 utiles)

| Colonne | Description |
|---|---|
| `GKGRECORDID` | Identifiant unique de l'enregistrement GKG |
| `DATE` | Date de publication (format `YYYYMMDDHHMMSS`, tronqué à 8 chiffres pour le parsing) |
| `SourceCommonName` | Nom du média source |
| `DocumentIdentifier` | URL de l'article |
| `Themes` / `V2Themes` | Thèmes GDELT détectés (ex : `ECON_SANCTIONS`, `GOV_CORRUPTION`) |
| `Locations` | Lieux mentionnés (format texte semi-structuré) |
| `Persons` | Personnes nommées dans l'article |
| `Organizations` | Organisations citées |
| `V2Tone` | Vecteur de tonalité multi-dimensions (6 scores dont positif, négatif, polarité) |
| `Quotations` | Citations directes extraites du texte |
| `AllNames` | Tous les noms propres détectés |

> Le fichier GKG utilise l'encodage `latin-1` et `on_bad_lines="skip"` car certains articles contiennent des caractères mal encodés.

In [ ]:

import zipfile
import io
import os
import time

from google.colab import drive

# Monter le Drive
drive.mount('/content/drive')
dossier = "/content/drive/MyDrive/hackathon_isheero"


df_clean = pd.read_csv(f"{dossier}/gdelt_benin_2025_clean.csv")
ids_benin = set(df_clean["GlobalEventID"].astype(str))
print(f" {len(ids_benin)} événements Bénin chargés")


print(" Chargement du masterfilelist...")
master = pd.read_csv(
    "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt",
    sep=" ", header=None,
    names=["taille", "md5", "url"],
    dtype=str
)


mois_2025 = [
    "202501", "202502", "202503", "202504",
    "202505", "202506", "202507", "202508",
    "202509", "202510", "202511", "202512"
]

master_2025_gkg = master[
    master["url"].str.contains("2025") &
    master["url"].str.contains("gkg")
]
print(f" Fichiers GKG 2025 : {len(master_2025_gkg)}")

colonnes_gkg = [
    "GKGRECORDID", "DATE", "SourceCollectionIdentifier",
    "SourceCommonName", "DocumentIdentifier", "Counts",
    "V2Counts", "Themes", "V2Themes", "Locations",
    "V2Locations", "Persons", "V2Persons", "Organizations",
    "V2Organizations", "V2Tone", "Dates", "GCAM",
    "SharingImage", "RelatedImages", "SocialImageEmbeds",
    "SocialVideoEmbeds", "Quotations", "AllNames",
    "Amounts", "TranslationInfo", "Extras"
]

resultats_gkg = []

for mois in mois_2025:
    fichiers_mois = master_2025_gkg[
        master_2025_gkg["url"].str.contains(mois)
    ]
    print(f"\n {mois} — {len(fichiers_mois)} fichiers...")
    compteur = 0

    for url in tqdm(fichiers_mois["url"], desc=mois):
        try:
            response = requests.get(url, timeout=30)
            zip_file = zipfile.ZipFile(io.BytesIO(response.content))

            df = pd.read_csv(
                zip_file.open(zip_file.namelist()[0]),
                sep="\t",
                header=None,
                names=colonnes_gkg,
                dtype=str,
                encoding="latin-1",
                on_bad_lines="skip"
            )

            df_gkg_bn = df[
                df["Locations"].str.contains("Benin", na=False) |
                df["V2Locations"].str.contains("Benin", na=False)
            ]

            if len(df_gkg_bn) > 0:
                resultats_gkg.append(df_gkg_bn)
                compteur += len(df_gkg_bn)

        except Exception as e:
            print(f" Erreur : {e}")
            continue

        time.sleep(0.1)

    print(f" {mois} — {compteur} entrées GKG Bénin trouvées")


df_gkg = pd.concat(resultats_gkg, ignore_index=True)
print(f"\n Total GKG brut : {len(df_gkg)}")


cols_utiles_gkg = [
    "GKGRECORDID", "DATE", "SourceCommonName",
    "DocumentIdentifier", "Themes", "V2Themes",
    "Locations", "Persons", "Organizations",
    "V2Tone", "Quotations", "AllNames"
]
df_gkg = df_gkg[cols_utiles_gkg]


df_gkg["DATE"] = pd.to_datetime(
    df_gkg["DATE"].str[:8],
    format="%Y%m%d",
    errors="coerce"
)


df_gkg = df_gkg[df_gkg["DATE"].dt.year == 2025]


df_gkg = df_gkg.drop_duplicates()
print(f" Après nettoyage : {len(df_gkg)} entrées")


chemin_gkg = f"{dossier}/gdelt_benin_2025_gkg.csv"
df_gkg.to_csv(chemin_gkg, index=False)
print(f" sauvegarde terminée : {chemin_gkg}")
